# Лабораторная работа 1.1. LU-разложение матриц

**Задание.** Реализовать алгоритм LU-разложения матриц (с выбором главного элемента) в виде программы.
Используя разработанное ПО, решить СЛАУ. Для матрицы СЛАУ вычислить определитель и обратную матрицу.

$$
\begin{cases}
x_1 - x_2 + x_3 - 2x_4 = -20 \\
-9x_1 - x_2 + x_3 + 8x_4 = 60 \\
-7x_1 + 8x_3 - 6x_4 = -60 \\
3x_1 - 5x_2 + x_3 - 6x_4 = -44
\end{cases}
$$

## Теория

Ищем разложение $PA = LU$, где $L$ — нижняя треугольная матрица с единицами на диагонали,
$U$ — верхняя треугольная, $P$ — матрица перестановок строк.

На $k$-м шаге метода Гаусса в качестве главного выбирается элемент столбца $k$
с максимальным модулем среди строк $k..n-1$: $|a_{pk}| = \max_{i \ge k} |a_{ik}|$.
Строки $k$ и $p$ переставляются (в $U$, в уже вычисленной части $L$ и в векторе перестановок).
Далее для $i > k$:

$$\mu_i = \frac{a_{ik}}{a_{kk}}, \qquad l_{ik} = \mu_i, \qquad a_{ij} := a_{ij} - \mu_i a_{kj},\ j = k..n-1.$$

**Решение СЛАУ** $Ax = b$ $\Rightarrow$ $LUx = Pb$:
1. прямая подстановка $Lz = Pb$: $\; z_i = (Pb)_i - \sum_{j<i} l_{ij} z_j$;
2. обратная подстановка $Ux = z$: $\; x_i = \dfrac{1}{u_{ii}}\left(z_i - \sum_{j>i} u_{ij} x_j\right)$.

**Определитель:** $\det A = (-1)^{s} \prod_i u_{ii}$, где $s$ — число перестановок строк.

**Обратная матрица:** $A A^{-1} = E$ — решаем $n$ систем $A x^{(j)} = e_j$ с уже готовым разложением;
столбцы $x^{(j)}$ образуют $A^{-1}$.

## Реализация

In [ ]:
def lu_decompose(A):
    """LU-разложение с выбором главного элемента по столбцу: PA = LU.

    Возвращает (L, U, perm, swaps), где perm[i] — номер исходной строки,
    стоящей на i-м месте, swaps — число перестановок строк.
    """
    n = len(A)
    U = [list(map(float, row)) for row in A]
    L = [[0.0] * n for _ in range(n)]
    perm = list(range(n))
    swaps = 0

    for k in range(n):
        # выбор главного элемента в столбце k
        p = max(range(k, n), key=lambda i: abs(U[i][k]))
        if abs(U[p][k]) < 1e-14:
            raise ValueError("Матрица вырождена")
        if p != k:
            U[k], U[p] = U[p], U[k]
            L[k], L[p] = L[p], L[k]
            perm[k], perm[p] = perm[p], perm[k]
            swaps += 1
        # исключение
        for i in range(k + 1, n):
            mu = U[i][k] / U[k][k]
            L[i][k] = mu
            for j in range(k, n):
                U[i][j] -= mu * U[k][j]

    for i in range(n):
        L[i][i] = 1.0
    return L, U, perm, swaps


def lu_solve(L, U, perm, b):
    """Решение Ax = b по готовому разложению PA = LU."""
    n = len(L)
    pb = [b[perm[i]] for i in range(n)]
    # прямая подстановка: Lz = Pb
    z = [0.0] * n
    for i in range(n):
        z[i] = pb[i] - sum(L[i][j] * z[j] for j in range(i))
    # обратная подстановка: Ux = z
    x = [0.0] * n
    for i in range(n - 1, -1, -1):
        x[i] = (z[i] - sum(U[i][j] * x[j] for j in range(i + 1, n))) / U[i][i]
    return x


def lu_det(U, swaps):
    det = (-1) ** swaps
    for i in range(len(U)):
        det *= U[i][i]
    return det


def lu_inverse(L, U, perm):
    n = len(L)
    cols = [lu_solve(L, U, perm, [1.0 if i == j else 0.0 for i in range(n)])
            for j in range(n)]
    return [[cols[j][i] for j in range(n)] for i in range(n)]  # транспонируем


def multiply(X, Y):
    """Произведение двух матриц (для проверки L * U = A)."""
    n, m, p = len(X), len(Y), len(Y[0])
    return [[sum(X[i][k] * Y[k][j] for k in range(m)) for j in range(p)]
            for i in range(n)]


def show(M, name=None, w=10, prec=4):
    if name:
        print(f"{name} =")
    for row in M:
        print("  [" + " ".join(f"{(0.0 if abs(v) < 1e-12 else v):{w}.{prec}f}" for v in row) + " ]")
    print()

## Исходные данные

In [9]:
A = [
    [ 1, -1, 1, -2],
    [-9, -1, 1,  8],
    [-7,  0, 8, -6],
    [ 3, -5, 1, -6],
]
b = [-20, 60, -60, -44]

show(A, "A", w=6, prec=0)
print("b =", b)

A =
  [     1     -1      1     -2 ]
  [    -9     -1      1      8 ]
  [    -7      0      8     -6 ]
  [     3     -5      1     -6 ]

b = [-20, 60, -60, -44]


## LU-разложение

In [ ]:
L, U, perm, swaps = lu_decompose(A)

show(L, "L")
show(U, "U")
print("Перестановка строк P:", perm, f"(число перестановок: {swaps})")
print()

PA = [list(map(float, A[perm[i]])) for i in range(len(A))]
show(multiply(L, U), "Проверка L·U")
show(PA, "P·A")

L =
  [    1.0000     0.0000     0.0000     0.0000 ]
  [   -0.3333     1.0000     0.0000     0.0000 ]
  [    0.7778    -0.1458     1.0000     0.0000 ]
  [   -0.1111     0.2083     0.1124     1.0000 ]

U =
  [   -9.0000    -1.0000     1.0000     8.0000 ]
  [    0.0000    -5.3333     1.3333    -3.3333 ]
  [    0.0000     0.0000     7.4167   -12.7083 ]
  [    0.0000     0.0000     0.0000     1.0112 ]

Перестановка строк P: [1, 3, 2, 0] (число перестановок: 2)

Проверка L·U =
  [   -9.0000    -1.0000     1.0000     8.0000 ]
  [    3.0000    -5.0000     1.0000    -6.0000 ]
  [   -7.0000     0.0000     8.0000    -6.0000 ]
  [    1.0000    -1.0000     1.0000    -2.0000 ]

P·A =
  [   -9.0000    -1.0000     1.0000     8.0000 ]
  [    3.0000    -5.0000     1.0000    -6.0000 ]
  [   -7.0000     0.0000     8.0000    -6.0000 ]
  [    1.0000    -1.0000     1.0000    -2.0000 ]



## Решение СЛАУ

In [11]:
x = lu_solve(L, U, perm, b)
for i, xi in enumerate(x, 1):
    print(f"x{i} = {xi:.6f}")

# невязка
r = [sum(A[i][j] * x[j] for j in range(4)) - b[i] for i in range(4)]
print("\nНевязка Ax - b:", [f"{v:.2e}" for v in r])

x1 = -4.000000
x2 = 0.000000
x3 = -8.000000
x4 = 4.000000

Невязка Ax - b: ['0.00e+00', '0.00e+00', '1.42e-14', '0.00e+00']


## Определитель

In [12]:
det = lu_det(U, swaps)
print(f"det A = {det:.6f}")

det A = 360.000000


## Обратная матрица

In [ ]:
A_inv = lu_inverse(L, U, perm)
show(A_inv, "A^-1", prec=6)

show(multiply(A, A_inv), "Проверка A · A^-1")

A^-1 =
  [  1.088889   0.022222  -0.111111  -0.222222 ]
  [ -0.194444  -0.111111   0.055556  -0.138889 ]
  [  1.694444   0.111111  -0.055556  -0.361111 ]
  [  0.988889   0.122222  -0.111111  -0.222222 ]

Проверка A · A^-1 =
  [    1.0000     0.0000     0.0000     0.0000 ]
  [    0.0000     1.0000     0.0000     0.0000 ]
  [    0.0000     0.0000     1.0000     0.0000 ]
  [    0.0000     0.0000     0.0000     1.0000 ]



## Проверка с помощью NumPy (необязательно)

In [14]:
try:
    import numpy as np
    An, bn = np.array(A, float), np.array(b, float)
    print("x      (numpy):", np.linalg.solve(An, bn))
    print("det    (numpy):", np.linalg.det(An))
    print("max |A^-1 - A^-1_numpy| =", np.abs(np.array(A_inv) - np.linalg.inv(An)).max())
except ImportError:
    print("NumPy не установлен — пропускаем сверку")

x      (numpy): [-4.00000000e+00  9.99200722e-16 -8.00000000e+00  4.00000000e+00]
det    (numpy): 360.00000000000006
max |A^-1 - A^-1_numpy| = 1.1102230246251565e-16


## Вывод

Реализовано LU-разложение с выбором главного элемента по столбцу ($PA = LU$).
С его помощью решена СЛАУ, вычислены определитель (как произведение диагонали $U$ с учётом знака перестановок)
и обратная матрица (решением $n$ систем с единичными правыми частями).
Невязки и проверка $A\cdot A^{-1} = E$ подтверждают корректность результатов.